# CellClick Repro Test

This notebook reproduces the common CellClick analysis flow by function calls only.

- Input: `D:\Projects\CellClick\data\3k_PBMCs.h5ad`
- Reference Comparison: `D:\Projects\CellClick\marker_ref\CellMarker\Gene Analysis\human.GeneWeight.mat`
- Cell Identification: `D:\Projects\CellClick\marker_ref\Other\Jin_PBMC_Cell Group.json`

In [1]:
from pathlib import Path
import json
import sys

project_dir = Path(r"D:\Projects\CellClick")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))


import scanpy as sc
from IPython.display import display

from scripts.adataProcessor import AdataProcessor

data_path = Path(r"D:\Projects\CellClick\data\3k_PBMCs.h5ad")
ref_compare_path = Path(r"D:\Projects\CellClick\marker_ref\CellMarker\Gene Analysis\human.GeneWeight.mat")
cell_identification_path = Path(r"D:\Projects\CellClick\marker_ref\Other\Jin_PBMC_Cell Group.json")

adata = sc.read_h5ad(data_path)
print(adata)
print('obs columns:', list(adata.obs.columns))
print('obsm keys:', list(adata.obsm.keys()))
print('uns keys:', list(adata.uns.keys()))


f:\Software\anaconda\envs\CellClick\lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
f:\Software\anaconda\envs\CellClick\lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
f:\Software\anaconda\envs\CellClick\lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
f:\Software\anaconda\envs\CellClick\lib\site-packages\anndata\utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warning

AnnData object with n_obs × n_vars = 2633 × 13714
    obs: 'leiden_CellClick', 'level_0', 'round_0'
    var: 'gene_ids', 'mt', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'leiden', 'log1p', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'X'
    obsp: 'connectivities', 'distances'
obs columns: ['leiden_CellClick', 'level_0', 'round_0']
obsm keys: ['X_pca', 'X_umap']
uns keys: ['hvg', 'leiden', 'log1p', 'neighbors', 'umap']


In [2]:
# Use the existing cluster annotation in the dataset.
groupby = 'leiden_CellClick'
annotation_series = adata.obs[groupby].copy()
cell_cluster = annotation_series.value_counts().idxmax()
cell_ids = annotation_series[annotation_series == cell_cluster].index.tolist()

print('groupby:', groupby)
print('selected cluster:', cell_cluster)
print('cell count:', len(cell_ids))

processor_ref = AdataProcessor(adata.copy())
processor_cell = AdataProcessor(adata.copy())

groupby: leiden_CellClick
selected cluster: 0
cell count: 1198


In [3]:
# Reference Comparison for all clusters: top scoring annotation table
# Uses the scoring core extracted from processor_ref.buildMarkerGeneScorePlot, so no figures are created.
# The raw reference hit is preserved, then standardized with Cell Ontology / OLS.
import json
import pandas as pd

from scripts.ontology_standardization import standardize_annotation_result

def _primary_mapping(standardization_result):
    standardized = standardization_result.get('standardized_annotation') or {}
    primary = standardized.get('primary_annotation') or {}
    records = primary.get('records') or []
    mapping = records[0].get('mapping', {}) if records else {}
    return primary, mapping

all_clusters = sorted(list(annotation_series.dropna().unique()), key=lambda value: str(value))
all_cluster_marker_scores = {}
marker_score_rows = []

for cluster in all_clusters:
    cluster_cell_ids = annotation_series[annotation_series == cluster].index.tolist()
    marker_gene_scores = processor_ref.MarkerGeneScores_cal(
        cellIDs=cluster_cell_ids,
        cellCluster=cluster,
        annotationSeries=annotation_series.copy(),
        marker_ref_path=str(ref_compare_path),
        markerGeneUsed=10,
        showNum=10,
        return_details=True,
    )
    all_cluster_marker_scores[str(cluster)] = marker_gene_scores

    scores = marker_gene_scores.get('scores', pd.Series(dtype=float))
    marker_series = marker_gene_scores.get('markerSeries', pd.Series(dtype=float))
    gene_weight_filtered = marker_gene_scores.get('geneWeight', pd.DataFrame())
    gene_weight_full = marker_gene_scores.get('geneWeightFull', pd.DataFrame())
    score_details = marker_gene_scores.get('score_details', pd.DataFrame())

    if len(scores) == 0:
        marker_score_rows.append({
            'cluster': str(cluster),
            'cell_count': len(cluster_cell_ids),
            'top_ref_cell_type': None,
            'standardized_cell_type': None,
            'ontology_id': None,
            'ontology_label': None,
            'mapping_status': 'unmapped',
            'tissue': None,
            'cell_type': None,
            'score': None,
            'annotation_confidence': None,
            'p_value': None,
            'q_value': None,
            'relative_margin': None,
            'overlap_num': 0,
            'overlap_genes': '',
            'cosg_unique_genes': ', '.join(map(str, marker_series.index)),
            'full_ref_gene_count': 0,
            'merged_raw_labels': '',
            'merge_reasons': '',
        })
        continue

    top_ref_cell_type = scores.index[0]
    tissue_cell = str(top_ref_cell_type).split('_', 1)
    tissue = tissue_cell[0] if len(tissue_cell) > 1 else ''
    cell_type = tissue_cell[1] if len(tissue_cell) > 1 else tissue_cell[0]

    overlap_genes = [
        gene for gene in marker_series.index
        if gene in gene_weight_filtered.columns and gene_weight_filtered.loc[top_ref_cell_type, gene] > 0
    ]
    overlap_gene_set = set(overlap_genes)
    cosg_unique_genes = [gene for gene in marker_series.index if gene not in overlap_gene_set]
    full_ref_gene_count = (
        int((gene_weight_full.loc[top_ref_cell_type] > 0).sum())
        if top_ref_cell_type in gene_weight_full.index else 0
    )

    raw_annotation = {
        'suggested_cell_type': str(top_ref_cell_type),
        'suggested_subtype': None,
        'alternative_annotations': [str(ref_cell_type) for ref_cell_type in scores.index[1:10]],
    }
    standardization_result = standardize_annotation_result(
        raw_annotation,
        [
            {
                'ref_cell_type': str(ref_cell_type),
                'cell_type': str(ref_cell_type).split('_', 1)[1] if '_' in str(ref_cell_type) else str(ref_cell_type),
                'score': float(score),
            }
            for ref_cell_type, score in scores.items()
        ],
        jaccard_threshold=0.6,
    )
    primary_annotation, primary_mapping = _primary_mapping(standardization_result)
    merge_reasons = '; '.join(
        [f"{item.get('raw_label')}: {item.get('reason')}" for item in (primary_annotation.get('merge_reasons') or [])]
    )
    top_summary = marker_gene_scores.get('top_candidate_summary') or {}

    marker_score_rows.append({
        'cluster': str(cluster),
        'cell_count': len(cluster_cell_ids),
        'top_ref_cell_type': str(top_ref_cell_type),
        'standardized_cell_type': primary_annotation.get('standard_label'),
        'ontology_id': primary_mapping.get('ontology_id'),
        'ontology_label': primary_mapping.get('ontology_label'),
        'mapping_status': primary_mapping.get('mapping_status'),
        'tissue': tissue,
        'cell_type': cell_type,
        'score': float(scores.iloc[0]),
        'annotation_confidence': top_summary.get('annotation_confidence'),
        'p_value': top_summary.get('p_value'),
        'q_value': top_summary.get('q_value'),
        'relative_margin': top_summary.get('relative_margin'),
        'overlap_num': len(overlap_genes),
        'overlap_genes': ', '.join(map(str, overlap_genes)),
        'cosg_unique_genes': ', '.join(map(str, cosg_unique_genes)),
        'full_ref_gene_count': full_ref_gene_count,
        'merged_raw_labels': ', '.join(map(str, primary_annotation.get('raw_labels', []))),
        'merge_reasons': merge_reasons,
    })

marker_score_table = pd.DataFrame(marker_score_rows)
display(marker_score_table)


**finished identifying marker genes by COSG**
**finished identifying marker genes by COSG**


D:\Projects\CellClick\scripts\adata_processor\annotation_eval.py:296: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  overlap_num.append(int(ref_gene_mask_full.reindex(markerSeries.index).fillna(False).sum()))


**finished identifying marker genes by COSG**
**finished identifying marker genes by COSG**


D:\Projects\CellClick\scripts\adata_processor\annotation_eval.py:296: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  overlap_num.append(int(ref_gene_mask_full.reindex(markerSeries.index).fillna(False).sum()))


**finished identifying marker genes by COSG**


D:\Projects\CellClick\scripts\adata_processor\annotation_eval.py:296: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  overlap_num.append(int(ref_gene_mask_full.reindex(markerSeries.index).fillna(False).sum()))


**finished identifying marker genes by COSG**


D:\Projects\CellClick\scripts\adata_processor\annotation_eval.py:296: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  overlap_num.append(int(ref_gene_mask_full.reindex(markerSeries.index).fillna(False).sum()))


,cluster,cell_count,top_ref_cell_type,standardized_cell_type,ontology_id,ontology_label,mapping_status,tissue,cell_type,score,annotation_confidence,p_value,q_value,relative_margin,overlap_num,overlap_genes,cosg_unique_genes,full_ref_gene_count,merged_raw_labels,merge_reasons
0,0,1198,circumventricular organ_naive thymus-derived C...,circumventricular organ naive thymus-derived C...,None,None,unmapped,circumventricular organ,"naive thymus-derived CD4-positive, alpha-beta ...",0.258623,0.853860,0.001996,0.002218,0.798704,8,"LDHB, CD3D, IL7R, CD3E, NOSIP, CCR7, LEF1, PIK...","LTB, IL32",119,circumventricular organ_naive thymus-derived C...,circumventricular organ_naive thymus-derived C...
1,1,642,blood_myeloid cell,blood myeloid cell,None,None,ols_unavailable,blood,myeloid cell,0.045725,0.563058,0.001996,0.002851,0.051026,4,"FCN1, LST1, S100A8, CD68","S100A9, CFD, AIF1, CST3, TYROBP, TYMP",18,blood_myeloid cell,blood_myeloid cell: new_group
2,2,410,spleen_natural killer cell,spleen natural killer cell,None,None,ols_unavailable,spleen,natural killer cell,0.096509,0.764470,0.001996,0.002218,0.162277,6,"NKG7, CST7, PRF1, GZMB, GNLY, CCL5","GZMA, GZMH, FGFBP2, CTSW",28,spleen_natural killer cell,spleen_natural killer cell: new_group
3,3,340,kidney_B cell,kidney venous blood vessel cell,CL:1000893,kidney venous blood vessel cell,ambiguous,kidney,B cell,0.184964,0.842813,0.001996,0.002218,0.278065,9,"CD79A, MS4A1, CD79B, TCL1A, VPREB3, HLA-DQA1, ...",LINC00926,650,kidney_B cell,kidney_B cell: new_group
4,4,31,circumventricular organ_myeloid dendritic cell,circumventricular organ myeloid dendritic cell,None,None,unmapped,circumventricular organ,myeloid dendritic cell,0.154287,0.651061,0.001996,0.002851,0.098600,3,"FCER1A, CLEC10A, CD1C","SERPINF1, ENHO, CLEC4C, CLIC2, LILRA4, RP6-91H...",10,circumventricular organ_myeloid dendritic cell,circumventricular organ_myeloid dendritic cell...
5,5,12,blood_megakaryocyte,blood megakaryocyte,None,None,ols_unavailable,blood,megakaryocyte,0.445029,0.853824,0.001996,0.003992,0.565837,6,"GP9, GNG11, PF4, ITGA2B, SPARC, PPBP","AP001189.4, SDPR, TMEM40, TREML1",16,blood_megakaryocyte,blood_megakaryocyte: new_group


In [15]:
# LLM-assisted cell type annotation for all clusters
import os
import pandas as pd

from scripts.llm_ann import evaluate_marker_gene_score_annotation

openlux_api_key = os.environ.get('OPENAI_API_KEY')
if not openlux_api_key:
    raise RuntimeError(
        'Set OPENAI_API_KEY before running this cell. '
        'Example in PowerShell: $env:OPENAI_API_KEY="your_api_key"'
    )

llm_annotation_rows = []
for cluster in all_clusters:
    marker_gene_scores = all_cluster_marker_scores[str(cluster)]
    try:
        llm_result = evaluate_marker_gene_score_annotation(
            marker_gene_scores,
            api_key=openlux_api_key,
            base_url='https://api.openlux.ai/v1',
            model='gpt-5.5',
            context={
                'dataset': str(data_path),
                'reference': str(ref_compare_path),
                'groupby': groupby,
                'selected_cluster': str(cluster),
                'selected_cell_count': int((annotation_series == cluster).sum()),
            },
            top_n=10,
            max_genes=50,
            standardize=True,
        )
        llm_annotation = llm_result.get('annotation') or {}
        if not isinstance(llm_annotation, dict):
            llm_annotation = {}
        top_summary = marker_gene_scores.get('top_candidate_summary') or {}
        standardized_annotation = llm_result.get('standardized_annotation') or {}
        primary_annotation = standardized_annotation.get('primary_annotation') or {}
        primary_records = primary_annotation.get('records') or []
        primary_mapping = primary_records[0].get('mapping', {}) if primary_records else {}
        llm_annotation_rows.append({
            'cluster': str(cluster),
            'suggested_cell_type': llm_annotation.get('suggested_cell_type'),
            'suggested_subtype': llm_annotation.get('suggested_subtype'),
            'confidence_score': llm_annotation.get('confidence_score'),
            'confidence_label': llm_annotation.get('confidence_label'),
            'alternative_annotations': llm_annotation.get('alternative_annotations'),
            'recommended_validation_genes': llm_annotation.get('recommended_validation_genes'),
            'annotation_confidence': top_summary.get('annotation_confidence'),
            'ontology_mapping_confidence': primary_mapping.get('mapping_confidence'),
            'ontology_id': primary_mapping.get('ontology_id'),
            'ontology_label': primary_mapping.get('ontology_label') or primary_annotation.get('standard_label'),
            'error': None,
        })
    except Exception as exc:
        llm_annotation_rows.append({
            'cluster': str(cluster),
            'suggested_cell_type': None,
            'suggested_subtype': None,
            'confidence_score': None,
            'confidence_label': None,
            'alternative_annotations': None,
            'recommended_validation_genes': None,
            'annotation_confidence': None,
            'ontology_mapping_confidence': None,
            'ontology_id': None,
            'ontology_label': None,
            'error': str(exc),
        })

llm_annotation_table = pd.DataFrame(llm_annotation_rows)
display(llm_annotation_table)

adata.obs['llm_cell_type_annotation_all'] = annotation_series.astype(str)
for row in llm_annotation_rows:
    label = row.get('suggested_cell_type')
    if label:
        adata.obs.loc[annotation_series.astype(str) == row['cluster'], 'llm_cell_type_annotation_all'] = label

display(adata.obs['llm_cell_type_annotation_all'].value_counts())


,cluster,suggested_cell_type,suggested_subtype,confidence_score,confidence_label,alternative_annotations,recommended_validation_genes,error
0,0,CD4-positive alpha-beta T cell,Naive T cell,0.259000,high,[{'cell_type': 'Central memory CD8-positive al...,"[CD4, CD8A, CD8B, TCF7, SELL, CD27, S1PR1, TRAC]",None
1,1,monocyte,FCN1+ S100A8/S100A9+ myeloid monocyte; classic...,0.045725,moderate,"[{'cell_type': 'blood myeloid cell', 'score': ...","[CD14, FCGR3A, FCN1, S100A8, S100A9, LST1, CST...",None
2,2,natural killer cell,cytotoxic NK cell,0.096509,high,"[{'cell_type': 'cytotoxic T cell', 'reason': '...","[KLRD1, KLRF1, KLRK1, NCR1, FCGR3A, CD3D, CD3E...",None
3,3,B cell,likely naive B cell; subtype assignment is mod...,0.184964,"high for B cell, moderate for naive subtype","[{'cell_type': 'B cell', 'reason': 'Generic B ...","[CD19, CD79A, MS4A1, CD79B, TCL1A, FCER2, BANK...",None
4,4,dendritic cell,"CD1C+ conventional/myeloid dendritic cell, wit...",0.580000,moderate,"[{'cell_type': 'plasmacytoid dendritic cell', ...","[CLEC4A, CLEC9A, IL3RA, GZMB, JCHAIN, PLD4, HL...",None
5,5,Megakaryocyte/platelet lineage cell,Blood megakaryocyte-like,0.445029,high,"[{'cell_type': 'Platelet', 'support': 'GP9, PF...","[ITGB3, TUBB1, FLI1, TREML1, SELP, CD63]",None


llm_cell_type_annotation_all
CD4-positive alpha-beta T cell         1198
monocyte                                642
natural killer cell                     410
B cell                                  340
dendritic cell                           31
Megakaryocyte/platelet lineage cell      12
Name: count, dtype: int64

In [5]:
# Cell Identification function
with open(cell_identification_path, 'r', encoding='utf-8') as f:
    marker_dict = json.load(f)

fig_cell, score_df, marker_cosg = processor_cell.buildCellScorePlot(
    cellIDs=cell_ids,
    annotation='selected',
    markerDict=marker_dict,
    title=f'Cell Identification - cluster {cell_cluster}',
    showNum=5,
    geneUsed=5,
    markerUsed=5,
)

display(fig_cell)
# display(score_df)
# display(marker_cosg)

**finished identifying marker genes by COSG**
**finished identifying marker genes by COSG**


D:\Projects\CellClick\scripts\adata_processor\annotation_eval.py:373: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [14]:
# LLM-assisted cell score validation for buildCellScorePlot
import os

import scripts.llm_ann as llm_ann

evaluate_cell_score_annotation = llm_ann.evaluate_cell_score_annotation

openlux_api_key = os.environ.get('OPENAI_API_KEY')
if not openlux_api_key:
    raise RuntimeError(
        'Set OPENAI_API_KEY before running this cell. '
        'Example in PowerShell: $env:OPENAI_API_KEY="your_api_key"'
    )

llm_cell_score_result = evaluate_cell_score_annotation(
    cell_score_plot_result=(fig_cell, score_df, marker_cosg),
    markerDict=marker_dict,
    api_key=openlux_api_key,
    base_url='https://api.openlux.ai/v1',
    model='gpt-5.5',
    context={
        'dataset': str(data_path),
        'reference': str(cell_identification_path),
        'selected_cluster': str(cell_cluster),
        'selected_cell_count': len(cell_ids),
        'analysis': 'buildCellScorePlot cell score validation',
    },
    top_n=5,
    max_genes=50,
)

llm_cell_score_annotation = llm_cell_score_result.get('annotation')
display(
    llm_cell_score_annotation
    if llm_cell_score_annotation is not None
    else llm_cell_score_result.get('raw_response')
)

{'suggested_reference_cell_type': 'CD4+ T cell',
 'annotation_supported': True,
 'confidence_score': 0.97,
 'confidence_label': 'high',
 'supporting_markers': ['CD3D',
  'CD3E',
  'CD3G',
  'IL7R',
  'CCR7',
  'LTB',
  'LEF1',
  'TCF7',
  'MAL',
  'TRAT1',
  'CD27',
  'CD28',
  'BCL11B',
  'LDHB',
  'IL32'],
 'conflicting_or_missing_markers': ['CD8+ T cell has a moderate cell score distribution and overlaps shared T cell markers including CD3D, CD3E, CD3G, IL32, CD2, LCK, CD6, and CD8B.',
  'CD4+ T cell reference-unique genes not present in the query top markers include INPP4B, AP3M2, DGKA, ITGA6, PRKCA, EEIG1, THEM4, CDC14A, PDE3B, and TRBC2.',
  'NK cell, RBC, and Neutrophil candidates have low cell scores and limited or no marker overlap.'],
 'alternative_reference_cell_types': [{'cell_type': 'CD8+ T cell',
   'support': 'moderate but clearly below CD4+ T cell',
   'mean_cell_score': 0.6702346600158031,
   'overlap_genes': ['CD3D',
    'CD3E',
    'IL32',
    'CD2',
    'LCK',
    '